# Appendix B — Computational model specification

Compare the canonical 150-tree, three-fold-calibrated configuration with the
earlier 500-tree configuration under the documented protocol. Small observed
differences justify a computational trade-off, not universal equivalence.


## 0. Configuration and feature reconstruction


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, tempfile
from pathlib import Path
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'mpl_cache'))

ROOT = Path.cwd()
while not (ROOT / 'config.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import config
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
from src.models import classification

RS = config.RANDOM_STATE



tr_def = classification.features_e2gp('train')
X_tr_def, cols_def = classification.e2gp_matrix(tr_def)
y_tr_def = tr_def['target'].to_numpy()

te_def = classification.features_e2gp('test', batch=4)
X_te_def, _ = classification.e2gp_matrix(te_def, cols_def)
rutas_b4 = config.BatchConfig(4)
meta_b4 = pd.read_csv(rutas_b4.test_metadata_clean_csv)[['object_id', 'true_target']]
tt_b4 = te_def[['object_id']].merge(meta_b4, on='object_id', how='left')['true_target'].to_numpy()
es_conocida = tt_b4 < 990
print(f"train {X_tr_def.shape} · test batch 4 {X_te_def.shape} · known {int(es_conocida.sum()):,}")


## 1. Compare the two specifications


In [ ]:
def evaluar_especificacion(n_estimators, cv_calibracion):
    rf = RandomForestClassifier(
        n_estimators=n_estimators, class_weight='balanced',
        random_state=RS, n_jobs=-1)
    mf1_cv = cross_val_score(
        rf, X_tr_def, y_tr_def,
        cv=StratifiedKFold(5, shuffle=True, random_state=RS),
        scoring='f1_macro', n_jobs=-1).mean()
    rf.fit(X_tr_def, y_tr_def)
    mf1_batch = f1_score(
        tt_b4[es_conocida], rf.predict(X_te_def[es_conocida]),
        average='macro', zero_division=0)
    calibrado = classification.calibrated_al_classifier(
        y=y_tr_def, n_estimators=n_estimators, cv=cv_calibracion,
        random_state=RS).fit(X_tr_def, y_tr_def)
    mf1_batch_cal = f1_score(
        tt_b4[es_conocida], calibrado.predict(X_te_def[es_conocida]),
        average='macro', zero_division=0)
    return mf1_cv, mf1_batch, mf1_batch_cal

comparacion_especificacion = []
for n_estimators, cv_cal in [(150, 3), (500, 5)]:
    mf1_cv, mf1_batch, mf1_batch_cal = evaluar_especificacion(n_estimators, cv_cal)
    comparacion_especificacion.append({
        'configuration': f'{n_estimators} trees / cv={cv_cal}',
        'MF1_CV_train': mf1_cv, 'MF1_batch4_RF': mf1_batch,
        'MF1_batch4_calibrado': mf1_batch_cal})
comparacion_especificacion = pd.DataFrame(comparacion_especificacion)
comparacion_especificacion.to_csv(
    config.DIR_PROCESSED / 'comparacion_especificacion_e2gp.csv', index=False)
display(comparacion_especificacion.round(4))


## Interpretation

Read validation and external results together with the estimator definitions
and runtime. This comparison supports the selected implementation; it is not
an independent validation of the final policy.
